# 08 — Final Evaluation

Menggabungkan hasil seluruh eksperimen menjadi tabel & visualisasi tunggal.

Eksperimen yang dibandingkan:
1. **Grid Search** baseline (notebook 03)
2. **Optuna**     (notebook 04)
3. **Univariate vs Multivariate** (notebook 05)
4. **±Lat/Long** (notebook 06)
5. **±NDVI** (notebook 07)

Output: `results/metrics/08_final_summary.csv` + plot.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src import config as C
sns.set_theme(style='whitegrid')

In [2]:
# Load semua hasil
grid   = pd.read_csv(C.METRICS_DIR / '03_grid_best_summary.csv')[['station', 'R2']].rename(columns={'R2': 'grid'})
optuna = pd.read_csv(C.METRICS_DIR / '04_optuna_summary.csv')[['station', 'R2']].rename(columns={'R2': 'optuna'})
uni_multi = pd.read_csv(C.METRICS_DIR / '05_univariate_vs_multivariate.csv')[['station', 'R2_uni', 'R2_multi']]
uni_multi = uni_multi.rename(columns={'R2_uni': 'univariate', 'R2_multi': 'multivariate'})
ll = pd.read_csv(C.METRICS_DIR / '06_lat_long_comparison.csv')
ll_pivot = ll.pivot(index='station', columns='variant', values='R2').reset_index()
ll_pivot.columns.name = None
ll_pivot = ll_pivot.rename(columns={'no_coords': 'multi_no_coords', 'with_coords': 'multi_with_coords'})
try:
    ndvi = pd.read_csv(C.METRICS_DIR / '07_ndvi_comparison.csv')[['station', 'R2_base', 'R2_ndvi']]
    ndvi = ndvi.rename(columns={'R2_base': 'no_ndvi', 'R2_ndvi': 'with_ndvi'})
except FileNotFoundError:
    ndvi = None
    print('NDVI belum dijalankan')

In [3]:
# Merge semua
summary = grid.merge(optuna, on='station').merge(uni_multi, on='station').merge(ll_pivot, on='station', how='left')
if ndvi is not None:
    summary = summary.merge(ndvi, on='station', how='left')
summary.to_csv(C.METRICS_DIR / '08_final_summary.csv', index=False)
summary.round(4)

,station,grid,optuna,univariate,multivariate,multi_no_coords,multi_with_coords,no_ndvi,with_ndvi
0,kelapa_gading,0.6344,6.460000e-01,0.6748,0.6355,0.7476,0.7189,0.6136,0.7741
1,bundaran_hi,0.5968,6.413000e-01,0.6094,0.5920,0.6760,0.7580,0.5173,0.4749
2,kebun_jeruk,0.5919,5.202000e-01,0.6877,0.4612,0.7134,0.7074,0.5513,0.6477
3,jagakarsa,0.4887,5.259000e-01,0.4596,0.3935,0.4780,0.5409,0.4890,0.5083
4,us_embassy_2,0.4158,-1.600000e-03,0.5517,0.3533,0.6091,0.6286,0.4219,0.5156
5,lubang_buaya,0.0171,-5.360000e-02,0.3536,-0.2836,0.4676,0.4510,-0.2324,-0.0387
6,us_embassy_1,-0.2173,-3.743227e+06,0.6241,-5.1318,0.6070,0.6448,-5973.5227,-1291.1111


In [ ]:
# Plot perbandingan R² per stasiun (semua eksperimen)
# Catatan: nilai R² < -1  di clip ke -1
# dan diberi anotasi "x" agar plot tetap terbaca tanpa skala dirusak outlier
import numpy as np

R2_FLOOR = -1.0  # batas bawah skala y-axis
long = summary.melt(id_vars='station', var_name='experiment', value_name='R2')

# Tandai failed (R² sangat negatif) sebelum di-clip
long['failed'] = long['R2'] < R2_FLOOR
long['R2_plot'] = long['R2'].clip(lower=R2_FLOOR)

fig, ax = plt.subplots(figsize=(14, 5.5))
sns.barplot(data=long, x='station', y='R2_plot', hue='experiment', ax=ax)
ax.axhline(0, color='black', linewidth=0.6)
ax.set_ylim(R2_FLOOR - 0.05, 1.0)
ax.set_ylabel('R² (test, skala asli µg/m³). Nilai < -1 di-clip ke -1.')
ax.set_title('Perbandingan R² semua eksperimen per stasiun')
ax.tick_params(axis='x', rotation=30)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Eksperimen')

# Tandai bar yang failed dengan tanda "x" di bawah
stations_order = list(summary['station'])
exps_order = [e for e in summary.columns if e != 'station']
n_exp = len(exps_order)
bar_w = 0.8 / n_exp
for i, station in enumerate(stations_order):
    for j, exp in enumerate(exps_order):
        row = long[(long['station'] == station) & (long['experiment'] == exp)]
        if len(row) and row.iloc[0]['failed']:
            x = i - 0.4 + bar_w * (j + 0.5)
            ax.text(x, R2_FLOOR + 0.02, 'x\nfail',
                    ha='center', va='bottom', fontsize=7, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig(C.FIG_DIR / '08_final_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

# Print mana saja yang failed (R² < -1)
failed_summary = long[long['failed']][['station', 'experiment', 'R2']].sort_values('R2')
if len(failed_summary):
    print('\nEksperimen yang failed (R² < -1, di-clip pada plot):')
    print(failed_summary.to_string(index=False))


In [5]:
# Eksperimen mana yang menang per stasiun?
non_id = [c for c in summary.columns if c != 'station']
summary['best_exp'] = summary[non_id].idxmax(axis=1)
summary['best_R2']  = summary[non_id].max(axis=1)
summary[['station', 'best_exp', 'best_R2']]

,station,best_exp,best_R2
0,kelapa_gading,with_ndvi,0.774067
1,bundaran_hi,multi_with_coords,0.757984
2,kebun_jeruk,multi_no_coords,0.713364
3,jagakarsa,multi_with_coords,0.540932
4,us_embassy_2,multi_with_coords,0.628592
5,lubang_buaya,multi_no_coords,0.467567
6,us_embassy_1,multi_with_coords,0.644839
